# Registre de modèles et promotion en production
## Reconnaissance du genre - noms propres ivoiriens | SIGAP / AFOR
**INP-HB · ESI · Stage 2A · 2025-2026**

1. **Paquets** : chaque candidat est mis sous la forme attendue par `adaptateurs.py`
   (un dossier avec `modele.json` et les fichiers du modèle), puis vérifié.
2. **Registre MLflow** `genre-noms-ivoiriens` : une version par candidat, avec ses scores
   de validation et de test (issus de `evaluation_finale.ipynb`).
3. **Alias** : `production` = modèle retenu par la règle de l'évaluation finale,
   `challenger` = meilleur suivant sur le test. Promouvoir un modèle = déplacer l'alias.
4. **Export** : le paquet `production` est copié dans `production/` (utilisé par l'app en
   local) et, si `PUBLIER = True`, publié dans le dossier `production/` du dépôt
   Hugging Face privé (utilisé par l'app en ligne).

Le registre reste local (`mlflow.db`). L'app n'a besoin ni de MLflow ni de TensorFlow.

## 0. Paramètres

In [ ]:
NOM_REGISTRE  = 'genre-noms-ivoiriens'
MLFLOW_URI    = 'sqlite:///mlflow.db'
FINAL_DIR     = 'runs/final'
RUNS_DIR      = 'runs'
PAQUETS_DIR   = 'paquets'
SEUIL_INCERTAIN = 0.70
PUBLIER       = False   # True : publie le paquet production sur Hugging Face
HF_MODEL_REPO = 'LeMental/gender-by-ivorian-names-model'
HF_TOKEN_FILE = '/mnt/c/Users/THIS/.cache/huggingface/token'   # session « hf auth login »

## 1. Configuration et données

In [ ]:
import os, re, json, glob, shutil, hashlib, pickle, tempfile, logging, warnings
# Processeur uniquement : sur GPU, les LSTM deviennent des opérations cuDNN que l'export
# ONNX ne sait pas convertir (CudnnRNNV3)
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
import numpy as np
import pandas as pd
import mlflow
from mlflow import MlflowClient
from adaptateurs import charger, clean_text, FICHE

warnings.filterwarnings('ignore')
logging.getLogger('mlflow').setLevel(logging.ERROR)
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('registre')
client = MlflowClient()
os.makedirs(PAQUETS_DIR, exist_ok=True)

df = pd.read_csv('dataset_nettoye.csv')
df['full_name'] = (df['NOM'].apply(clean_text) + ' ' + df['PRENOMS'].fillna('').apply(clean_text)).str.strip()
df = df[df['full_name'] != ''].reset_index(drop=True)
df['label'] = (df['GENRE'] == 'M').astype(int)
sp = pd.read_csv('split_reference.csv')
emp = df['full_name'].map(lambda s: hashlib.sha1(s.encode()).hexdigest()[:10]).values
assert len(sp) == len(df) and (sp['empreinte'].values == emp).all(), "split_reference.csv ne correspond pas"
idx_va = df.index[sp['ensemble'].values == 'val']
va_texts, y_va = list(df.loc[idx_va, 'full_name']), df.loc[idx_va, 'label'].values

ev = json.load(open(os.path.join(FINAL_DIR, 'evaluation_finale.json')))
scores = {r['modele']: r for r in ev['resultats']}
print(f"Évaluation finale : choix = {ev['choix']} ({ev['raison']})")

## 2. Paquets des candidats

| Clé | Modèle | Données d'entraînement |
|---|---|---|
| `lr` | régression logistique (modèle de production) | toutes les données |
| `v5` | modèle par mot, meilleure graine sur la validation | train |
| `v4` | CNN + BiLSTM + MHA réentraîné, meilleure graine sur la validation | train |
| `v4_origine` | CNN + BiLSTM + MHA du notebook d'origine (historique) | train d'origine |

Pour les réseaux, chaque paquet est vérifié : ses prédictions sur la validation doivent être
identiques à celles enregistrées par l'évaluation finale.

In [ ]:
def ecrire_fiche(dossier, **fiche):
    fiche.setdefault('seuil_incertain', SEUIL_INCERTAIN)
    with open(os.path.join(dossier, FICHE), 'w', encoding='utf-8') as f:
        json.dump(fiche, f, indent=1, ensure_ascii=False)


def meilleure_graine(arch):
    """Graine dont l'accuracy de validation est la meilleure (choix sur la validation)."""
    accs = {}
    for f in glob.glob(os.path.join(FINAL_DIR, f'{arch}_seed*.npz')):
        s = int(re.search(r'seed(\d+)', f).group(1))
        accs[s] = ((np.load(f)['val'] > 0.5) == y_va).mean()
    return max(accs, key=accs.get), accs


PAQUETS = {}

# ── Régression logistique de production : toutes les données ─────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union
import joblib, sklearn

d = os.path.join(PAQUETS_DIR, 'lr'); os.makedirs(d, exist_ok=True)
lr = make_pipeline(make_union(TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 6), sublinear_tf=True),
                              TfidfVectorizer(analyzer='word', token_pattern=r'\S+')),
                   LogisticRegression(C=2, max_iter=3000))
lr.fit(df['full_name'], df['label'])
joblib.dump(lr, os.path.join(d, 'pipeline.joblib'), compress=3)
ecrire_fiche(d, type='lr', nom='Régression logistique', fichiers=dict(pipeline='pipeline.joblib'),
             entrainement=f"toutes les données ({len(df):,} noms)", scikit_learn=sklearn.__version__,
             config="TF-IDF n-grammes de caractères 1-6 + mots, C = 2", cle_evaluation='regression_logistique')
PAQUETS['lr'] = d

# ── v5 : ONNX déjà exporté par optimisation.ipynb ────────────────────────────
s, accs = meilleure_graine('v5')
d = os.path.join(PAQUETS_DIR, 'v5'); os.makedirs(d, exist_ok=True)
shutil.copy(os.path.join(RUNS_DIR, f'model_v5_seed{s}.onnx'), os.path.join(d, 'modele.onnx'))
shutil.copy(os.path.join(RUNS_DIR, f'model_v5_seed{s}_vocab.json'), os.path.join(d, 'vocab.json'))
ecrire_fiche(d, type='v5_onnx', nom='Modèle par mot (v5)', fichiers=dict(onnx='modele.onnx', vocab='vocab.json'),
             entrainement='train', graine=s, cle_evaluation='v5')
PAQUETS['v5'] = d
print(f"v5 : graine {s} retenue (val acc {({k: round(v, 4) for k, v in accs.items()})})")

# ── v4 réentraîné : export ONNX depuis Keras ─────────────────────────────────
s, accs = meilleure_graine('v4')
d = os.path.join(PAQUETS_DIR, 'v4'); os.makedirs(d, exist_ok=True)
import tensorflow as tf, tf2onnx
keras_v4 = tf.keras.models.load_model(os.path.join(RUNS_DIR, f'model_v4_seed{s}.keras'), compile=False)
vocab = json.load(open(os.path.join(RUNS_DIR, f'model_v4_seed{s}_vocab.json')))
tf2onnx.convert.from_keras(keras_v4, opset=15, output_path=os.path.join(d, 'modele.onnx'),
                           input_signature=[tf.TensorSpec((None, vocab['max_len']), tf.int32, name='input')])
json.dump(dict(char_to_index=vocab['char_to_index'], max_len=vocab['max_len']),
          open(os.path.join(d, 'vocab.json'), 'w'))
ecrire_fiche(d, type='v4_onnx', nom='CNN + BiLSTM + MHA (v4)', fichiers=dict(onnx='modele.onnx', vocab='vocab.json'),
             entrainement='train', graine=s, cle_evaluation='v4')
PAQUETS['v4'] = d
print(f"v4 : graine {s} retenue (val acc {({k: round(v, 4) for k, v in accs.items()})})")

# ── v4 d'origine (modèle déployé avant ce travail) ───────────────────────────
if os.path.exists('best_gender_model.onnx'):
    d = os.path.join(PAQUETS_DIR, 'v4_origine'); os.makedirs(d, exist_ok=True)
    shutil.copy('best_gender_model.onnx', os.path.join(d, 'modele.onnx'))
    json.dump(dict(char_to_index=pickle.load(open('char_to_index.pkl', 'rb')),
                   max_len=pickle.load(open('max_len.pkl', 'rb'))), open(os.path.join(d, 'vocab.json'), 'w'))
    ecrire_fiche(d, type='v4_onnx', nom="CNN + BiLSTM + MHA (v4 d'origine)",
                 fichiers=dict(onnx='modele.onnx', vocab='vocab.json'),
                 entrainement="train d'origine (train_model.ipynb)", cle_evaluation=None)
    PAQUETS['v4_origine'] = d

In [ ]:
# Vérification de chaque paquet via l'adaptateur
for cle, d in PAQUETS.items():
    m = charger(d)
    p = m.predict_proba(va_texts)
    ligne = f"{cle:<11} {m.nom:<36} val acc = {((p > 0.5) == y_va).mean():.4f}"
    fiche = json.load(open(os.path.join(d, FICHE)))
    if 'graine' in fiche:
        ref = np.load(os.path.join(FINAL_DIR, f"{cle}_seed{fiche['graine']}.npz"))['val']
        ecart = float(np.abs(p - ref).max())
        assert ecart < 1e-3, f"{cle} : prédictions différentes de l'évaluation ({ecart:.2e})"
        ligne += f" | écart avec l'évaluation finale = {ecart:.1e}"
    if cle == 'lr':
        ligne += " (entraînée aussi sur la validation : score non comparable)"
    print(ligne)

## 3. Enregistrement dans le registre MLflow

Chaque paquet devient une version de `genre-noms-ivoiriens` (modèle `pyfunc` qui embarque
`adaptateurs.py`). Une version identique (même empreinte de fichiers) n'est pas
réenregistrée si le notebook est relancé.

In [ ]:
class ModeleGenre(mlflow.pyfunc.PythonModel):
    """Enveloppe MLflow : entrée = noms bruts, sortie = P(Masculin)."""

    def load_context(self, context):
        from adaptateurs import charger
        self.modele = charger(context.artifacts['paquet'])

    def predict(self, context, model_input, params=None):
        from adaptateurs import clean_text
        noms = model_input.iloc[:, 0] if hasattr(model_input, 'iloc') else model_input
        return self.modele.predict_proba([clean_text(n) for n in noms])


def empreinte(dossier):
    h = hashlib.sha1()
    for f in sorted(os.listdir(dossier)):
        h.update(f.encode()); h.update(open(os.path.join(dossier, f), 'rb').read())
    return h.hexdigest()[:12]


try:
    existantes = client.search_model_versions(f"name='{NOM_REGISTRE}'")
except Exception:
    existantes = []
deja = {(v.tags.get('cle'), v.tags.get('empreinte')): v.version for v in existantes}

VERSIONS = {}
for cle, d in PAQUETS.items():
    fiche, emp = json.load(open(os.path.join(d, FICHE))), empreinte(d)
    if (cle, emp) in deja:
        VERSIONS[cle] = deja[(cle, emp)]
        print(f"{cle:<11} déjà enregistré : version {VERSIONS[cle]}")
        continue
    sc = scores.get(fiche.get('cle_evaluation')) or {}
    with mlflow.start_run(run_name=f'registre_{cle}'):
        mlflow.log_params({k: v for k, v in fiche.items() if k not in ('fichiers',)})
        mlflow.log_metrics({f'test_{k}': sc[k] for k in ('acc', 'f1', 'auc', 'precision_si_reponse')
                            if k in sc})
        info = mlflow.pyfunc.log_model(
            name='modele', python_model=ModeleGenre(), artifacts={'paquet': d},
            code_paths=['adaptateurs.py'], registered_model_name=NOM_REGISTRE,
            pip_requirements=[f'scikit-learn=={sklearn.__version__}', 'onnxruntime', 'numpy', 'joblib'])
    v = str(info.registered_model_version)
    VERSIONS[cle] = v
    tags = dict(cle=cle, empreinte=emp, type=fiche['type'], entrainement=fiche['entrainement'])
    tags.update({f'test_{k}': f"{sc[k]:.4f}" for k in ('acc', 'f1', 'auc') if k in sc})
    for k, val in tags.items():
        client.set_model_version_tag(NOM_REGISTRE, v, k, str(val))
    client.update_model_version(NOM_REGISTRE, v, description=f"{fiche['nom']} - entraînement : {fiche['entrainement']}")
    print(f"{cle:<11} enregistré : version {v}")

## 4. Alias : `production` et `challenger`

- `production` : le modèle choisi par la règle de `evaluation_finale.ipynb` (régression
  logistique sauf modèle significativement meilleur). Si ce choix n'a pas de paquet
  (ensemble, boosting), la régression logistique est gardée et un message l'indique.
- `challenger` : la meilleure accuracy de test parmi les autres versions évaluées.

`promouvoir(version)` permet ensuite de changer de modèle de production à la main.

In [ ]:
CORRESPONDANCE = {'regression_logistique': 'lr', 'v5': 'v5', 'v4': 'v4'}


def promouvoir(version, alias='production'):
    client.set_registered_model_alias(NOM_REGISTRE, alias, str(version))
    print(f"alias '{alias}' -> version {version}")


cle_prod = CORRESPONDANCE.get(ev['choix'])
if cle_prod is None:
    print(f"Choix '{ev['choix']}' sans paquet : la régression logistique reste en production")
    cle_prod = 'lr'
promouvoir(VERSIONS[cle_prod], 'production')

autres = {c: scores[k]['acc'] for k, c in CORRESPONDANCE.items() if c != cle_prod and k in scores and c in VERSIONS}
if autres:
    promouvoir(VERSIONS[max(autres, key=autres.get)], 'challenger')

rows = []
for v in client.search_model_versions(f"name='{NOM_REGISTRE}'"):
    alias = [a for a in client.get_registered_model(NOM_REGISTRE).aliases
             if client.get_registered_model(NOM_REGISTRE).aliases[a] == v.version]
    rows.append(dict(version=int(v.version), cle=v.tags.get('cle'), alias=', '.join(alias),
                     test_acc=v.tags.get('test_acc', '-'), entrainement=v.tags.get('entrainement')))
display(pd.DataFrame(rows).sort_values('version'))

## 5. Export du paquet `production`

Le paquet de la version portant l'alias `production` est copié dans `production/` (dossier
local ignoré par git, lu par l'app), avec son numéro de version. Avec `PUBLIER = True`, il est
aussi publié dans le dossier `production/` du dépôt Hugging Face privé : l'app en ligne le
charge au prochain redémarrage.

In [ ]:
def exporter_production(publier=PUBLIER):
    v = client.get_model_version_by_alias(NOM_REGISTRE, 'production')
    with tempfile.TemporaryDirectory() as tmp:
        mlflow.artifacts.download_artifacts(f"models:/{NOM_REGISTRE}@production", dst_path=tmp)
        src = os.path.dirname(glob.glob(os.path.join(tmp, '**', FICHE), recursive=True)[0])
        if os.path.exists('production'):
            shutil.rmtree('production')
        shutil.copytree(src, 'production')
    fiche = json.load(open(os.path.join('production', FICHE)))
    fiche.update(version_registre=int(v.version), registre=NOM_REGISTRE)
    json.dump(fiche, open(os.path.join('production', FICHE), 'w'), indent=1, ensure_ascii=False)
    m = charger('production')
    print(f"production/ : {m.nom}, version {v.version} | test : 'kouassi marie' -> "
          f"P(M) = {m.predict_proba(['kouassi marie'])[0]:.3f}")
    if publier:
        from huggingface_hub import HfApi
        token = open(HF_TOKEN_FILE).read().strip()
        HfApi(token=token).upload_folder(
            folder_path='production', path_in_repo='production', repo_id=HF_MODEL_REPO,
            delete_patterns='*', commit_message=f"Production : {m.nom}, version {v.version}")
        print(f"Publié dans {HF_MODEL_REPO}/production")


exporter_production()